# Simple Benchmark Model Training


This Notebook adapted from the [genomic bechmarks](https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks.git) explicitly `genomic_benchmarks/experiments/tf_cnn_experiments/demo_coding_vs_intergenomic_seqs.ipynb`. This notebook trains and stores the sample model, trained on the benchmark dataset.

In [1]:
import warnings
warnings.filterwarnings('ignore')

# TF CNN Classifier

To run this notebook on an another benchmark, use

```
papermill utils/tf_cnn_classifier.ipynb tf_cnn_experiments/[DATASET NAME].ipynb -p DATASET [DATASET NAME]
```

In [2]:
DATASET = 'demo_coding_vs_intergenomic_seqs'
VERSION = 0
BATCH_SIZE = 64
EPOCHS = 10

In [3]:
# Parameters
DATASET = "demo_coding_vs_intergenomic_seqs"


In [4]:
print(DATASET, VERSION, BATCH_SIZE, EPOCHS)

demo_coding_vs_intergenomic_seqs 0 64 10


# Data download

In [5]:
from pathlib import Path
import tensorflow as tf
import tensorflow_addons as tfa

import numpy as np
from genomic_benchmarks.loc2seq import download_dataset
from genomic_benchmarks.data_check import is_downloaded, info
from genomic_benchmarks.models.tf import vectorize_layer
from genomic_benchmarks.models.tf import get_basic_cnn_model_v0 as get_model
import pandas as pd

if not is_downloaded(DATASET):
    download_dataset(DATASET)

2025-07-18 14:26:55.013705: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
info(DATASET)

Dataset `demo_coding_vs_intergenomic_seqs` has 2 classes: coding_seqs, intergenomic_seqs.

All lengths of genomic intervals equals 200.

Totally 100000 sequences have been found, 75000 for training and 25000 for testing.


,train,test
coding_seqs,37500,12500
intergenomic_seqs,37500,12500


## TF Dataset object

In [7]:
SEQ_PATH = Path("data/original_genomic_benchmark")
CLASSES = [x.stem for x in (SEQ_PATH/'train').iterdir() if x.is_dir()]
NUM_CLASSES = len(CLASSES)

train_dset = tf.keras.preprocessing.text_dataset_from_directory(
    SEQ_PATH / 'train',
    batch_size=BATCH_SIZE,
    class_names=CLASSES)

Found 75000 files belonging to 2 classes.


In [8]:
if NUM_CLASSES > 2:
    train_dset = train_dset.map(lambda x, y: (x, tf.one_hot(y, depth=NUM_CLASSES)))

## Text vectorization

In [19]:
test = train_dset.map(lambda x, y: x)
for i in test:
    print(i)

tf.Tensor(
[b'GTAGAAGGAGTTTACTGGGTCCAGTCCACACACAAGTGGAGGAGATTACAAAAGGGTATGAATGCCAGGAGACAGGAGCCTCTGGGAATCATGTCAGTAGCTGTCTATCACAAAGATGAAGAAACTGAGGCACCGAAAATTTATGGATTTGGCCAAAGATCCCACAGTTAATAAGCTAAGCAACCTGGGAATTAGATACA'
 b'AGACAGCGAGAAGAGCTTCTGTCTCGAACCTTCACCACTAACGACTCTGACACCACCATACCAATGGACGAATCACTGCAGTTTAACTCCTCCCTCCAGAAAGTTCACAACGGCATGGATGACCTCATTTTAGATGGGCACAATATTTTAGATGGACTGAGGACCCAGAGACTGACCTTGAAGGGGACTCAGAAGAAGAT'
 b'GCCAGGTCCATGAAGAATTTATAAATAACATGGCTAATACTACAGTTGAAAGTTTGATACAAAAATTTGCTGAGTCAAAAGGCACTGGGAAGGAAAGACCTGGCCTCATTGAATTTGAAGAATGTGACACTGCTAGTGCAGTTGAAGGTATAAAACCAAGGAAAAGAAAGACCTTTGCTTTGCCAGGAATCATTAAAAAG'
 b'TTGATGGCACCAAGTAGCAAACGTGCTGTGGAGACTAGATCTCACTCTCAAATCTGTCCTGGGATGCAAGTGCCTCTACGAATGTGGTGCAGAAGCTACAAATGTGGTGCAGAAGACAGCTTCAAAGAGGAAAGGAATATAGGTGCCAAGAGCCTGCAATCCTTCCCAAACAGCCCTTTGTTCTGTGCTCAAGTCAACCC'
 b'GAATGTCATCATGTTCCTGGGAGATGGGATGGGTGTCTCCACAGTGACGGCTGCCCGCATCCTCAAGGGTCAGCTCCACCACAACCCTGGGGAGGAGACCAGGCTGGAGATGGACAAGTTCCCCTTCGTGGCCCTCTCCAAGACGTACAACACCAATGCCCAGGTC

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



tf.Tensor(
[b'AACATGGGAGCACCCAGATTCATAAAACAACTTCTTAGAGACCTATAAAGAAACTGAGATAGTGGCACAGTAATAGTGGAAGACTGCAATACCCCCTTAACAGATGGTGTTAGGGGGATCTTGTCATCACACTGTTAGCTTTGTTAAGCTGTCATGTAGACTTGATTGTATAGTTGATTTATTTTTTGTTTACATTAACT'
 b'GTAGTTCGTTGGTTTTCTTTCCCCTCATCCTTTTGCCTGCTCCCGGCGAGGGGTGGCTTTGATTTCGGCGATGAGCTCCCAGAAAGGCAACGTGGCTCGTTCCAGACCTCAGAAGCACCAGAATACGTTTAGCTTCAAAAATGACAAGTTCGATAAAAGTGTGCAGACCAAGAAAATTAATGCAAAACTTCATGATGGAG'
 b'AGGAAATTCAGCAGGGCGAAGACGCCTTCCCACCTAGCTCTCCTCTCTTTGCAGAGCCATACAAAGTTACTAGCAAAGAAGATAAGTTATCAAGTCGTATTCAGAGTATGCTTGGAAACTACGATGAAATGAAGGATTTCATAGGAGACAGATCTATACCAAAGCTTGTTGCAATTCCCAAGCCTACAGTACCACCATCA'
 b'GAAAGTCTCTACCGGCTGTTACCTCAGACAACCCCGGAGAACATCAACAAGAACTTTGAGCAGTACACGTTGGACCCAGGGACACGGTACCCAAACCTCAACTCACACTGTGTCAGGCCCCATCAGGTGAAGCATTTGTATATCACTAAGGAGCTGGAGCACTACCCTCTCGAGAGACTGGGCTCGGTGAGGAGATCTGT'
 b'TACTGATGGTAAAACTAATTTGATCAGGTAACACATTCATCAGGTATGTCCATCCACCTTCCATGGAATGCTGCAATTGAAAACTAAGGCCTTGTTTTGTTATTTGCTAACTCTGTGACTTAGAGCCACTGATTCACATTTTTGGAGTCTCAGTCTCCTTGTGTCT

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



tf.Tensor(
[b'CCCTGTCTCAACTAAAAAAAGAAATACAAAAATTAGCTTGGTGTGGCTGGGCATGGTGGCTCATCCCTGTAATCCCAGCACTTTGGGAGGCCGAGGCGGGCAGATCATGAGGTCAGGAGATCAAGACCATCCTGGCTAACATGGTGAAATGAAACCCCGTCTCTACTAAAAATACAAAAAATTAGCCAGGCACGGTGGCA'
 b'AGATCGGAAAATCTCCTAAGTCAACTATTAAACCCATCTCTAGGAGGAAGAAGTTATCTGGGATAAGGTACTTCTTCTTTGGAAGTTTTTCCATCCTCAGAATAGACATGCAAGAAAAGCTTCATGTATTTTCAGTTGATGGTGTCTTGAATAAAATGAACAAAAGACAGACATAATTTTTAGAAGGCAAAATTTCCATT'
 b'GGTCTGCGGGCCTCACGCCTTCTCTGGTCACTGCTGAGGACTCCTCTCTGGAGTGTAGCAAGGCTGAGGACTCTGATGCCACAGGTCACGAGTGGAAGTTGGAGGGGGCACTCTCAGAGGAACCGCGGGGCCCCGAGTTGGGCTCTCTGGAACTTGTGGAGGACGACACAGTGGATTCAGATGCCACAAATGGCCTTATC'
 b'ATTTGAGTTAATATTTTTGTATGTTCTGGGGTAAGGGTTCGAATTGATTATTTTGCAAGTGGTGATCCACGTGTACGTTGTCAACCCAGTTTGTTCAAAGATTGTCTCTTCCTCATTGAATTGCACATGGCACCACTGTAAGAATCCATTGACTATAGATACATAGTTTTATATATGGACTCTCAATTCTCTTCCAACAA'
 b'AGGCAGAGCCTCCTCATCCTGCCAGTCTATTTAACTGTTCTCTGGAGTCCTTCCTGTCCCAGGAGATGAGGGAGTGGAGGGAGGGCCCTGAGAAGGTTCCCACACTGGGTCAACGAAGAAGGTGCCTGATCGGGCAAATCCCAGGTGGTTGGGGAAGGCAGGCAGA

In [9]:
vectorize_layer.adapt(train_dset.map(lambda x, y: x))
VOCAB_SIZE = len(vectorize_layer.get_vocabulary())
vectorize_layer.get_vocabulary()

['', '[UNK]', 'a', 't', 'g', 'c']

In [10]:
def vectorize_text(text, label):
  text = tf.expand_dims(text, -1)
  return vectorize_layer(text)-2, label

train_ds = train_dset.map(vectorize_text)

## Model training

In [11]:
model = get_model(NUM_CLASSES, VOCAB_SIZE)

In [21]:
history = model.fit(
    train_ds,
    epochs=EPOCHS)

model.save("simple_benchmark_model_saved")

Epoch 1/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2298 - binary_accuracy: 0.9085 - f1_score: 0.9008
Epoch 2/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2264 - binary_accuracy: 0.9098 - f1_score: 0.9016
Epoch 3/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2257 - binary_accuracy: 0.9104 - f1_score: 0.9026
Epoch 4/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2237 - binary_accuracy: 0.9110 - f1_score: 0.9033
Epoch 5/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2197 - binary_accuracy: 0.9125 - f1_score: 0.9056
Epoch 6/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2190 - binary_accuracy: 0.9124 - f1_score: 0.9050
Epoch 7/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2168 - binary_accuracy: 0.9142 - f1_score: 0.9069
Epoch 8/10
1172/1172 [==============================] - 13s 11ms/step - loss: 0.2162 - bin

INFO:tensorflow:Assets written to: simple_benchmark_model_saved/assets
